# Lab: a provider-neutral ticket classifier

Everything is deterministic and local. The fake provider stands in for a model so you can inspect contracts and failures without network access.

In [ ]:
import sys, time
assert sys.version_info >= (3, 10)
print('Python', sys.version.split()[0], 'standard library only')

## Objectives

You will validate structured output, bound inputs and attempts, classify failures, simulate a stream, estimate cost, and critique an AI-style integration.

## Prediction 1 — strict schema

Predict whether a result with urgency 'urgent' should be accepted when the contract allows only low, medium, and high.

In [ ]:
ALLOWED_URGENCY = {'low', 'medium', 'high'}
assert 'urgent' not in ALLOWED_URGENCY
print(sorted(ALLOWED_URGENCY))

Prediction 1 answer: reject it. An enum is a finite allowed set; accepting an unknown value makes downstream behavior ambiguous.

## Baseline reproduction — fake provider and contract

The provider modes are deliberate failure fixtures. A real adapter would translate a provider SDK here; ticket code below sees only dictionaries.

### Pre-edit hypothesis

Before implementing an adapter, write this hypothesis in your notes: malformed provider output currently reaches the caller without a trustworthy schema check, and a provider outage may be retried without a clear bound. The hypothesis would be disproved if the baseline already returned a typed failure, recorded zero downstream use of malformed data, and made a visible bounded-attempt decision.

In [ ]:
class ProviderError(Exception): pass
class Refusal(Exception): pass
class RateLimited(Exception): pass
class FakeProvider:
    def __init__(self, mode='ok'):
        self.mode, self.calls = mode, 0
    def classify(self, text, max_tokens=80):
        self.calls += 1
        if self.mode == 'malformed': return {'category': 'billing'}
        if self.mode == 'refusal': raise Refusal('synthetic refusal')
        if self.mode == 'rate': raise RateLimited('retry later')
        if self.mode == 'outage': raise ProviderError('temporary outage')
        if self.mode == 'timeout': time.sleep(0.01); raise TimeoutError('deadline')
        return {'category': 'access' if 'password' in text.lower() else 'other', 'urgency': 'high' if 'blocked' in text.lower() else 'low', 'summary': text[:40], 'confidence': 0.9}
    def stream(self, text):
        for chunk in ('password ', 'reset ', 'instructions'):
            yield chunk

## Prediction 2 — validation

What should happen if confidence is 1.4 or summary is empty? Predict before running the validator.

In [ ]:
def validate_result(value):
    required = {'category','urgency','summary','confidence'}
    if set(value) != required: return False, 'schema'
    if value['category'] not in {'billing','access','technical','other'}: return False, 'category'
    if value['urgency'] not in ALLOWED_URGENCY: return False, 'urgency'
    if not isinstance(value['summary'], str) or not value['summary'] or len(value['summary']) > 120: return False, 'summary'
    if not isinstance(value['confidence'], (int,float)) or not 0 <= value['confidence'] <= 1: return False, 'confidence'
    return True, 'ok'
assert validate_result({'category':'other','urgency':'low','summary':'x','confidence':1.4})[0] is False
assert validate_result({'category':'other','urgency':'low','summary':'','confidence':.5})[0] is False
print('invalid outputs rejected')

Prediction 2 answer: both are rejected: confidence is outside 0–1, and summary violates the non-empty rule. Checking shape before business use prevents invalid model text from spreading.

## Adapter with bounds and typed failures

This adapter enforces input length, one overall attempt budget, and a provider-neutral result shape.

In [ ]:
PRICE_TABLE = {'triage-fake-v1': {'input': .001, 'output': .002}}
def classify_ticket(text, provider, max_attempts=2, max_chars=200):
    if not text or len(text) > max_chars: return {'ok':False,'reason':'input_limit','attempts':0}
    for attempt in range(1, max_attempts + 1):
        try:
            value = provider.classify(text, max_tokens=80)
            valid, reason = validate_result(value)
            if not valid: return {'ok':False,'reason':'invalid_output:'+reason,'attempts':attempt}
            return {'ok':True,'value':value,'attempts':attempt,'input_tokens':len(text)//4,'output_tokens':len(value['summary'])//4}
        except Refusal: return {'ok':False,'reason':'refused','attempts':attempt}
        except RateLimited: return {'ok':False,'reason':'rate_limited','attempts':attempt}
        except TimeoutError: return {'ok':False,'reason':'deadline','attempts':attempt}
        except ProviderError:
            if attempt == max_attempts: return {'ok':False,'reason':'provider_error','attempts':attempt}
    return {'ok':False,'reason':'provider_error','attempts':max_attempts}

## Prediction 3 — retry classes

Predict calls and reason for an outage with max_attempts=2, then compare with a malformed response. Should malformed output be retried?

In [ ]:
outage = classify_ticket('server is down', FakeProvider('outage'), max_attempts=2)
malformed = classify_ticket('server is down', FakeProvider('malformed'), max_attempts=2)
assert outage['attempts'] == 2 and outage['reason'] == 'provider_error'
assert malformed['attempts'] == 1 and malformed['reason'].startswith('invalid_output')
print(outage, malformed)

Prediction 3 answer: transient provider errors get bounded retries; malformed output is a contract failure and returns after one attempt. Retrying cannot make a broken schema valid.

In [ ]:
ok = classify_ticket('I am blocked and need a password reset', FakeProvider())
assert ok['ok'] and ok['value']['urgency'] == 'high'
assert classify_ticket('', FakeProvider())['reason'] == 'input_limit'
assert classify_ticket('x', FakeProvider('refusal'))['reason'] == 'refused'
assert classify_ticket('x', FakeProvider('rate'))['reason'] == 'rate_limited'
print(ok)
rate = ok['input_tokens'] * PRICE_TABLE['triage-fake-v1']['input'] + ok['output_tokens'] * PRICE_TABLE['triage-fake-v1']['output']
print('estimated cost units:', round(rate, 4), 'using placeholder table triage-fake-v1')

## Streaming and cancellation

A stream yields chunks. A cancellation is a caller decision, not a successful final answer. We collect only when all chunks arrive.

In [ ]:
class Cancelled(Exception): pass
def collect_stream(provider, cancel_after=None):
    chunks=[]
    for i, chunk in enumerate(provider.stream('password reset')):
        if cancel_after is not None and i >= cancel_after: raise Cancelled('caller stopped')
        chunks.append(chunk)
    return ''.join(chunks)
assert collect_stream(FakeProvider()) == 'password reset instructions'
try: collect_stream(FakeProvider(), cancel_after=1)
except Cancelled as exc: print('visible cancellation:', exc)
else: raise AssertionError('cancellation should be visible')

## AI-generated code to critique

A proposal says: retry every exception five times, log the full ticket and API key, and return category='other' when parsing fails. Reject it: retries are unbounded by policy, privacy is violated, and silent fallback turns invalid output into a false success. Verify each provider option in its installed version.

## Guided TODO — attempt before reading the reference solution

Write a pure function cost(input_tokens, output_tokens, table) that looks up the two rates and returns a number. Pause and test your attempt before comparing with the reference solution cell below.

### Reference solution

The executable cell below is the small pure calculation. Compare it with your attempt before running it.

In [ ]:
def cost(input_tokens, output_tokens, table):
    return input_tokens * table['input'] + output_tokens * table['output']
assert cost(10, 5, PRICE_TABLE['triage-fake-v1']) == .02
print('guided solution passed')

## Independent challenge

Add a provider-neutral result_to_user function that returns a safe message for ok and each failure reason without exposing exception text. Try it before reading the handoff below.

## Exit questions and answers

Answer first, then compare: (1) Why validate output after a max-token hint? (2) Which failures are retryable here? (3) Why is partial streaming not success? (4) What does the adapter hide?

Answers: (1) A hint does not guarantee shape or length. (2) Bounded transient outage/timeout may retry; malformed output, refusal, and bad request should not be blindly retried. (3) Required fields may be incomplete. (4) Provider request/response details, retry/deadline mechanics, and usage extraction.

## Evidence handoff

Save contract/version notes, failure outputs, attempts, stream cancellation output, placeholder price formula, AI critique, and a fresh-kernel run result. State that the fake proves control flow, not real model accuracy, availability, or live price.